[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day13-end-to-end-inference-anatomy.ipynb)

# Day 13 — Putting It Together: End-to-End Inference Anatomy
**Tag:** CPU-OK (T4 optional, faster). ~45 min.

Today you instrument one request end to end and close the model-vs-measurement loop from Days 9-11.


In [ ]:
%pip install -q transformers torch
# Expected: install completes silently.

## 0. Setup: SmolLM2-135M + device

Same model as Day 1. ~270 MB download, runs fine on CPU.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M").to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"device={device}  params={n_params:,}")
# Expected: device=cpu (or cuda on T4), params ~ 134,515,008

## 1. Build a 512-token prompt

Repeat a paragraph until the prompt is ~512 tokens. Long enough that prefill is measurable, short enough for CPU.

In [ ]:
para = ("The transformer architecture processes sequences in parallel during training, "
        "but inference is autoregressive: each new token requires a full forward pass. ")
text = para * 40
ids = tok(text, return_tensors="pt").input_ids.to(device)
n_prompt = ids.shape[1]
print(f"prompt tokens: {n_prompt}")
# Expected: prompt tokens: ~500-560 (tokenizer-dependent; anything in 450-600 is fine)

## 2. Instrument the four stages

Time tokenize / prefill / decode / detokenize separately with `perf_counter`.
Decode uses a manual KV-cache loop (your Day 8 machinery) so the per-step cost is isolated from prefill.

In [ ]:
N_DECODE = 100

# --- stage 1: tokenize (re-time it in isolation) ---
t0 = time.perf_counter()
ids_t = tok(text, return_tensors="pt").input_ids.to(device)
t_tokenize = (time.perf_counter() - t0) * 1000

# --- stage 2: prefill (one forward pass over the full prompt) ---
with torch.no_grad():
    t0 = time.perf_counter()
    out = model(ids_t, use_cache=True)
    t_prefill = (time.perf_counter() - t0) * 1000
    past = out.past_key_values
    next_id = out.logits[:, -1:].argmax(-1)

# --- stage 3: decode (100 steps, cache reused) ---
step_times = []
generated = []
with torch.no_grad():
    for _ in range(N_DECODE):
        t0 = time.perf_counter()
        out = model(next_id, past_key_values=past, use_cache=True)
        step_times.append((time.perf_counter() - t0) * 1000)
        past = out.past_key_values
        next_id = out.logits[:, -1:].argmax(-1)
        generated.append(next_id.item())

# --- stage 4: detokenize ---
t0 = time.perf_counter()
decoded = tok.decode(generated)
t_detok = (time.perf_counter() - t0) * 1000

ms_per_token = sum(step_times) / len(step_times)
print(f"tokenize: {t_tokenize:.1f} ms | prefill: {t_prefill:.1f} ms | "
      f"decode mean: {ms_per_token:.1f} ms/token | detok: {t_detok:.1f} ms")
print("sample:", decoded[:80].replace("\n", " "))
# Expected on CPU: tokenize 5-20 ms, prefill 0.5-2 s, decode 30-80 ms/token, detok < 1 ms.
# On T4 divide everything by ~5-10x.

## 3. The latency ledger

One request's byte budget, with a share column. Decode should dominate — if it doesn't, your prefill prompt is too short relative to the decode length.

In [ ]:
total = t_tokenize + t_prefill + ms_per_token * N_DECODE + t_detok
rows = [
    ("tokenize", t_tokenize),
    ("prefill", t_prefill),
    (f"decode x{N_DECODE}", ms_per_token * N_DECODE),
    ("detokenize", t_detok),
]
print(f"{'stage':<14}{'ms':>10}{'share':>8}")
for name, ms in rows:
    print(f"{name:<14}{ms:>10.1f}{ms/total*100:>7.1f}%")
print(f"{'TOTAL':<14}{total:>10.1f}")

# TTFT / TPOT in the SLO vocabulary (scaled to this tiny model)
ttft = t_tokenize + t_prefill + ms_per_token
print(f"\nTTFT ~ {ttft:.1f} ms   TPOT ~ {ms_per_token:.1f} ms/token")
print(f"decode share of total: {ms_per_token*N_DECODE/total*100:.0f}%")
# Expected: decode >= 90% of total. TTFT dominated by prefill; TPOT = steady-state step cost.

## 4. Reconcile with the cost model (Day 10)

Predicted decode ms/token = weight bytes / achieved bandwidth. On CPU, "bandwidth" is DRAM (~20-40 GB/s effective for this access pattern) and launch overhead inflates the gap — the reconciliation below is the deliverable, not the raw number.

In [ ]:
weight_bytes = n_params * 2  # fp16-equivalent traffic per step (fp32 model: x2)
# Effective bandwidth guesses: CPU DRAM ~25 GB/s, T4 HBM ~200 GB/s effective for tiny GEMMs
bw = 200e9 if device == "cuda" else 25e9
predicted_ms = weight_bytes / bw * 1000
gap = (ms_per_token - predicted_ms) / predicted_ms * 100
print(f"weight bytes/step : {weight_bytes/1e9:.2f} GB")
print(f"assumed BW        : {bw/1e9:.0f} GB/s  ({device})")
print(f"predicted         : {predicted_ms:.1f} ms/token")
print(f"measured          : {ms_per_token:.1f} ms/token")
print(f"gap               : {gap:+.0f}%")
print("\nReconciliation (write 2-3 lines in your notes): which of attention-FLOPs-at-",
      n_prompt, "ctx / KV traffic / launch overhead explains YOUR gap?")
# Expected: gap within ~20-40% on CPU (launch overhead dominates at 135M scale);
# closer on T4. If gap > 100%, your bw guess is off -- tune it, that IS the exercise.

## 5. The mental model, in your own words (fill-in)

Copy this into `week02/mental-model.md` in your lab repo (local). The packet's paragraph is the answer key — rewrite it from memory first, then compare.

In [ ]:
%%writefile mental-model.md
# The inference expert's mental model (Day 13)

## The one paragraph
<!-- LLM inference is a memory-bandwidth problem wearing a compute costume... (your words) -->

## The two formulas
- FLOPs per decode token ~ 2 x params =
- Decode ms/token ~ weight bytes / bandwidth =

## Why prefill is compute-bound and decode is memory-bound
<!-- one sentence -->
# Expected: file mental-model.md written. Fill the blanks from memory BEFORE peeking at the packet.

## 6. Close out weeks 1-2 (local)

Review `week01/` end to end, fix anything that bit-rotted since Day 1, then tag the repo. Run this on your laptop/VM, not in Colab:

```bash
cd ~/llm-inference-lab
git add -A && git commit -m "week 1-2: foundations complete" || true
git tag -a week1-complete -m "Weeks 1-2 foundations done"
git push --tags
```

## Done

Fill the "What to measure" table in the Day 13 packet PDF with your numbers from cells 6-8. Tomorrow (Day 14): the throughput-vs-batch-size curve — your first mini-project.